# Chapter 2 — Exercises

Use this notebook only for the chapter exercises. Attempt each exercise before checking references or solutions.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "exercises" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)

print(f"pandas: {pd.__version__}")
print(f"Matplotlib: {plt.matplotlib.__version__}")
print(f"scikit-learn: {sklearn.__version__}")

pandas: 3.0.6
Matplotlib: 3.11.2
scikit-learn: 1.9.1


## Core: data loading, preprocessing, models, cross-validation

Reused from `chapters/02-end-to-end-project.ipynb` as a starting point for the exercises below.

In [2]:
import tarfile
import urllib.request

def load_housing():
    tarball_path = PROJECT_ROOT / "datasets" / "housing.tgz"
    if not tarball_path.is_file():
        tarball_path.parent.mkdir(parents=True, exist_ok=True)
        url = "https://github.com/ageron/data/raw/main/housing.tgz"
        urllib.request.urlretrieve(url, tarball_path)
        with tarfile.open(tarball_path) as tar:
            tar.extractall(path=tarball_path.parent, filter="data")
    return pd.read_csv(tarball_path.parent / "housing" / "housing.csv")

housing_full = load_housing()


In [3]:
from sklearn.model_selection import train_test_split

# Stratified split on income_cat so the test set represents the full income distribution.
housing_full["income_cat"] = pd.cut(
    housing_full["median_income"],
    bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
    labels=[1, 2, 3, 4, 5],
)

strat_train_set, strat_test_set = train_test_split(
    housing_full, test_size=0.2, stratify=housing_full["income_cat"], random_state=RANDOM_STATE
)

for set_ in (strat_train_set, strat_test_set):
    set_.drop("income_cat", axis=1, inplace=True)

housing = strat_train_set.drop("median_house_value", axis=1)
housing_labels = strat_train_set["median_house_value"].copy()


In [4]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler


class ClusterSimilarity(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=10, gamma=1.0, random_state=None):
        self.n_clusters = n_clusters
        self.gamma = gamma
        self.random_state = random_state

    def fit(self, X, y=None, sample_weight=None):
        self.k_means_ = KMeans(n_clusters=self.n_clusters, random_state=self.random_state)
        self.k_means_.fit(X, sample_weight=sample_weight)
        return self

    def transform(self, X):
        return rbf_kernel(X, self.k_means_.cluster_centers_, gamma=self.gamma)

    def get_feature_names_out(self, names=None):
        return [f"Cluster {i} similarity" for i in range(self.n_clusters)]


def column_ratio(X):
    return X[:, [0]] / X[:, [1]]


def ratio_name(function_transformer, feature_names_in):
    return ["ratio"]


def ratio_pipeline():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        FunctionTransformer(column_ratio, feature_names_out=ratio_name),
        StandardScaler(),
    )


def log_pipeline():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        FunctionTransformer(np.log, feature_names_out="one-to-one"),
        StandardScaler(),
    )


cat_pipeline = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore"),
)

cluster_simil = ClusterSimilarity(n_clusters=10, gamma=1.0, random_state=RANDOM_STATE)

default_num_pipeline = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())

preprocessing = ColumnTransformer([
    ("bedrooms", ratio_pipeline(), ["total_bedrooms", "total_rooms"]),
    ("rooms_per_house", ratio_pipeline(), ["total_rooms", "households"]),
    ("people_per_house", ratio_pipeline(), ["population", "households"]),
    ("log", log_pipeline(), ["total_bedrooms", "total_rooms", "population", "households", "median_income"]),
    ("geo", cluster_simil, ["latitude", "longitude"]),
    ("cat", cat_pipeline, make_column_selector(dtype_include="object")),
], remainder=default_num_pipeline)


In [5]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeRegressor

# Linear Regression baseline
lin_reg = make_pipeline(preprocessing, LinearRegression())
lin_reg.fit(housing, housing_labels)
lin_rmse = root_mean_squared_error(housing_labels, lin_reg.predict(housing))
print(f"Linear Regression train RMSE: {lin_rmse:,.2f}")  # underfits

# Decision Tree
tree_reg = make_pipeline(preprocessing, DecisionTreeRegressor(random_state=RANDOM_STATE))
tree_reg.fit(housing, housing_labels)
tree_rmse = root_mean_squared_error(housing_labels, tree_reg.predict(housing))
print(f"Decision Tree train RMSE: {tree_rmse:,.2f}")  # overfits (0 error on training set)

# Cross-validation to get a realistic estimate of generalization error
tree_rmses = -cross_val_score(
    tree_reg, housing, housing_labels, scoring="neg_root_mean_squared_error", cv=10
)
print("Decision Tree CV RMSE:")
print(pd.Series(tree_rmses).describe())

# Random Forest
forest_reg = make_pipeline(preprocessing, RandomForestRegressor(random_state=RANDOM_STATE))
forest_rmses = -cross_val_score(
    forest_reg, housing, housing_labels, scoring="neg_root_mean_squared_error", cv=10
)
print("Random Forest CV RMSE:")
print(pd.Series(forest_rmses).describe())


Linear Regression train RMSE: 68,972.89


Decision Tree train RMSE: 0.00


Decision Tree CV RMSE:
count       10.000000
mean     67013.360949
std       1460.198570
min      64289.376198
25%      66776.146282
50%      67086.216281
75%      68140.275029
max      68659.294290
dtype: float64


Random Forest CV RMSE:
count       10.000000
mean     47124.604437
std       1069.311372
min      45292.329302
25%      46712.106520
50%      47172.209883
75%      47561.377695
max      49354.705514
dtype: float64


In [6]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from scipy.stats import randint

full_pipeline = Pipeline([
    ("preprocessing", preprocessing),
    ("random_forest", RandomForestRegressor(random_state=RANDOM_STATE)),
])

param_distribs = {
    "preprocessing__geo__n_clusters": randint(3, 50),
    "random_forest__max_features": randint(2, 20),
}

rnd_search = RandomizedSearchCV(
    full_pipeline,
    param_distributions=param_distribs,
    n_iter=10,
    cv=3,
    scoring="neg_root_mean_squared_error",
    random_state=RANDOM_STATE,
)
rnd_search.fit(housing, housing_labels)

final_model = rnd_search.best_estimator_
feature_importances = final_model["random_forest"].feature_importances_
print(sorted(zip(feature_importances.round(3), final_model["preprocessing"].get_feature_names_out()), reverse=True))


[(np.float64(0.188), 'log__median_income'), (np.float64(0.078), 'cat__ocean_proximity_INLAND'), (np.float64(0.061), 'bedrooms__ratio'), (np.float64(0.058), 'rooms_per_house__ratio'), (np.float64(0.046), 'people_per_house__ratio'), (np.float64(0.042), 'geo__Cluster 30 similarity'), (np.float64(0.025), 'geo__Cluster 9 similarity'), (np.float64(0.024), 'geo__Cluster 36 similarity'), (np.float64(0.021), 'geo__Cluster 18 similarity'), (np.float64(0.019), 'geo__Cluster 32 similarity'), (np.float64(0.019), 'geo__Cluster 3 similarity'), (np.float64(0.018), 'geo__Cluster 25 similarity'), (np.float64(0.016), 'geo__Cluster 26 similarity'), (np.float64(0.016), 'geo__Cluster 1 similarity'), (np.float64(0.014), 'geo__Cluster 7 similarity'), (np.float64(0.014), 'geo__Cluster 10 similarity'), (np.float64(0.013), 'geo__Cluster 34 similarity'), (np.float64(0.013), 'geo__Cluster 19 similarity'), (np.float64(0.013), 'geo__Cluster 12 similarity'), (np.float64(0.013), 'geo__Cluster 0 similarity'), (np.float

In [7]:
X_test = strat_test_set.drop("median_house_value", axis=1)
y_test = strat_test_set["median_house_value"].copy()

final_predictions = final_model.predict(X_test)
final_rmse = root_mean_squared_error(y_test, final_predictions)
print(f"Final model test RMSE: {final_rmse:,.2f}")


Final model test RMSE: 41,423.08


### Exercise 1

Try SVM regressor with different hyperparameters.